In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV
import pandas as pd
import numpy as np

In [ ]:
df_train = pd.read_csv('../input/titanic/train.csv')
y_train = df_train.loc[:, 'Survived'].to_numpy()
df_train = df_train.drop(columns='Survived')
df_test = pd.read_csv('../input/titanic/test.csv')
df = pd.concat([df_train, df_test], axis=0)
df = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])
df

In [ ]:
obj_cols = df.loc[:, df.dtypes == object]
s1 = df.mean(numeric_only=True)
s2 = obj_cols.mode().loc[0]
s = pd.concat([s1, s2], axis=0)
df = df.fillna(value=s)
df

In [ ]:
dummies = pd.get_dummies(obj_cols)
df = pd.concat([df, dummies], axis=1)
df = df.drop(obj_cols, axis=1)
df

In [ ]:
x = df.to_numpy()
scaler = StandardScaler()
scaler.fit(x)
x = scaler.transform(x)
x_train = x[0: y_train.shape[0]]
x_test = x[y_train.shape[0]: ]

In [ ]:
'''param_grid = {'kernel': ['linear', 'poly', 'rbf'], 'C': [0.1, 0.5, 1, 2.5, 5, 10, 20, 50, 100]}
svm = SVC()
svm = GridSearchCV(svm, param_grid, cv=5)
svm.fit(x_train, y_train)
print(svm.best_score_)
print(svm.best_params_)'''

In [ ]:
svm = SVC(kernel='rbf', C=1)
svm.fit(x_train, y_train)
svm_pred = svm.predict(x_train)
print(accuracy_score(svm_pred, y_train))

In [ ]:
'''param_grid = {'learning_rate': [0.01, 0.1], 'n_estimators': [100, 200, 300]}
g_boost = GradientBoostingClassifier()
g_boost = GridSearchCV(g_boost, param_grid, cv=5)
g_boost.fit(x_train, y_train)
print(g_boost.best_score_)
print(g_boost.best_params_)'''

In [ ]:
g_boost = GradientBoostingClassifier(learning_rate=0.1, n_estimators=200)
g_boost.fit(x_train, y_train)
g_boost_pred = g_boost.predict(x_train)
print(accuracy_score(g_boost_pred, y_train))

In [ ]:
sample_sub = pd.read_csv('../input/titanic/gender_submission.csv')
svm_pred = svm.predict(x_test)
g_boost_pred = g_boost.predict(x_test)
sample_sub.loc[:, 'Survived'] = np.around((svm_pred + g_boost_pred) / 2).astype(int)
sample_sub.to_csv('submission.csv',index=False)